<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 08. SAC

## Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor

- **원 논문:** [Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor](https://proceedings.mlr.press/v80/haarnoja18b.html)
- **저자 / 발표:** Tuomas Haarnoja, Aurick Zhou, Pieter Abbeel, and Sergey Levine · ICML (2018)
- **난이도 / 예상 시간:** 고급 · 약 70분
- **선수 지식:** maximum-entropy RL, reparameterized Gaussian, actor-critic
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** squashed Gaussian actor와 soft actor objective를 로컬 continuous-action replay에서 학습한다.

**축소하거나 생략한 부분:** 논문은 MuJoCo continuous control과 twin-style off-policy updates를 평가한다. 축소판은 one-step continuous contextual control로 entropy/Q trade-off를 격리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.2, Eq. (1) | maximum-entropy objective | reward와 expected policy entropy를 함께 최대화한다. |
| §4.1, Eq. (2)–(3), Eq. (7)–(8) | soft Bellman backup and Q loss | soft value가 들어가는 critic target 구조를 연결한다. |
| §4.2, Eq. (10)–(12), Algorithm 1 | KL policy update and reparameterized stochastic actor | tanh Gaussian sample로 alpha log pi - Q를 최소화한다. |

## 핵심 재현

        Eq. (11)–(12)의 reparameterized stochastic actor를 사용해
        $J_\pi=\mathbb E[\alpha\log\pi(a|s)-Q(s,a)]$를 최소화합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(481)
np.random.seed(481)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
J_\pi=\mathbb E_{s,\epsilon}[\alpha\log\pi_\theta(a|s)-Q_\phi(s,a)],\qquad a=\tanh(\mu_\theta(s)+\sigma_\theta(s)\epsilon)
$$

- **기호와 역할:** $\alpha$는 entropy temperature, $Q_\phi$는 soft critic, tanh Jacobian은 bounded action의 log-prob를 보정합니다.
- **shape/state 계약:** `one-hot state [B, S] + action [B, 1] → Q [B, 1]; logp [B]`
- **논문 위치:** `§3.2, Eq. (1)`의 **maximum-entropy objective**
- **코드 Task:** reparameterized action/log-prob, critic MSE와 alpha logpi-Q를 구현합니다.
- **학습·평가 흐름:** off-policy batch → critic update → rsample actor update → mean/diversity 평가
- **원문 대비 한계:** 논문은 MuJoCo continuous control과 twin-style off-policy updates를 평가한다. 축소판은 one-step continuous contextual control로 entropy/Q trade-off를 격리한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class SoftActorCriticAgent(nn.Module):
    """SAC의 핵심 학습·평가 경로. 입출력 계약: one-hot state [B, S] + action [B, 1] → Q [B, 1];
    logp [B]."""

    def __init__(
        self,
        actor_network,
        critic_network,
        actor_optimizer,
        critic_optimizer,
        temperature=0.08,
    ):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.actor_network = actor_network
        self.critic_network = critic_network
        self.actor_optimizer = actor_optimizer
        self.critic_optimizer = critic_optimizer
        self.temperature = temperature

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"action_dim": 1, "temperature": self.temperature}

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: one-hot state [B, S] + action [B, 1] → Q
        [B, 1]; logp [B]."""
        action, log_probability, mean_action = self.actor_network.sample(state)
        encoded_state = encode_device(state)
        q_value = self.critic_network(torch.cat([encoded_state, action], dim=1))
        return (action, log_probability, mean_action, q_value)

    def loss(self, state, replay_action, reward):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        encoded_state = encode_device(state)
        predicted_q = self.critic_network(
            torch.cat([encoded_state, replay_action], dim=1)
        )
        critic_loss = F.mse_loss(predicted_q, reward)
        sampled_action, log_probability, _, q_value = self.forward(state)
        del sampled_action
        actor_loss = (self.temperature * log_probability - q_value.squeeze(1)).mean()
        return (critic_loss, actor_loss)

    def update(self, state, replay_action, reward):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        encoded_state = encode_device(state)
        predicted_q = self.critic_network(
            torch.cat([encoded_state, replay_action], dim=1)
        )
        critic_loss = F.mse_loss(predicted_q, reward)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        action, log_probability, _, q_value = self.forward(state)
        del action
        actor_loss = (self.temperature * log_probability - q_value.squeeze(1)).mean()
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        return (float(critic_loss.detach()), float(actor_loss.detach()))

    def evaluate(self, state, target_action):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            _, _, mean_action, q_value = self.forward(state)
            action_mse = F.mse_loss(mean_action, target_action)
        return {"action_mse": float(action_mse), "mean_q": float(q_value.mean())}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§4.2, Eq. (10)–(12), Algorithm 1` — **KL policy update and reparameterized stochastic actor**
- **구현 이유:** tanh Gaussian sample로 alpha log pi - Q를 최소화한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class GaussianActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_states, 32), nn.ReLU())
        self.mean = nn.Linear(32, 1)
        self.logstd = nn.Linear(32, 1)

    def sample(self, s):
        h = self.body(encode_device(s))
        mean = self.mean(h)
        logstd = self.logstd(h).clamp(-4, 1)
        std = logstd.exp()
        normal = torch.distributions.Normal(mean, std)
        u = normal.rsample()
        a = torch.tanh(u)
        logp = normal.log_prob(u) - torch.log(1 - a.square() + 1e-6)
        return a, logp.sum(1), torch.tanh(mean)


actor = GaussianActor().to(DEVICE)
a, lp, m = actor.sample(torch.arange(n_states))
assert (
    a.shape == (n_states, 1)
    and lp.shape == (n_states,)
    and float(a.detach().abs().max()) <= 1
)

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§4.1, Eq. (2)–(3), Eq. (7)–(8)` — **soft Bellman backup and Q loss**
- **구현 이유:** soft value가 들어가는 critic target 구조를 연결한다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
critic = nn.Sequential(nn.Linear(n_states + 1, 48), nn.ReLU(), nn.Linear(48, 1)).to(
    DEVICE
)
oq = torch.optim.Adam(critic.parameters(), lr=0.006)
op = torch.optim.Adam(actor.parameters(), lr=0.004)
alpha = 0.08
rng = np.random.default_rng(482)
replay_s = torch.arange(n_states).repeat_interleave(80)
replay_a = torch.tensor(
    rng.uniform(-1, 1, size=(len(replay_s), 1)), dtype=torch.float32
)
target = (q_star.argmax(1).float() / max(1, n_actions - 1) * 2 - 1)[:, None]
replay_target = target[replay_s]
replay_r = -(replay_a - replay_target).square()
replay_s, replay_a, replay_r, target = ACCELERATOR.move(
    replay_s, replay_a, replay_r, target
)
gen = torch.Generator().manual_seed(483)
history = []

portfolio_agent = SoftActorCriticAgent(actor, critic, op, oq, alpha)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in actor.parameters()]
)
for _ in range(360):
    index = torch.randint(len(replay_s), (64,), generator=gen).to(DEVICE)
    history.append(
        portfolio_agent.update(replay_s[index], replay_a[index], replay_r[index])
    )
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in actor.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§3.2, Eq. (1)` — **maximum-entropy objective**
- **구현 이유:** reward와 expected policy entropy를 함께 최대화한다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(torch.arange(n_states), target)
assert portfolio_metrics

with torch.no_grad():
    samples = []
for _ in range(60):
    samples.append(actor.sample(torch.arange(n_states))[0].detach())
samples = torch.stack(samples)
mean_action = actor.sample(torch.arange(n_states))[2].detach()
mse = float(F.mse_loss(mean_action, target))
diversity = float(samples.std(0).mean())
print(f"mean-action MSE={mse:.4f}, stochastic std={diversity:.3f}")
assert mse < 0.45 and diversity > 0
h = np.asarray(history)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(target.detach().cpu(), label="optimal proxy")
axes[0].plot(mean_action.detach().cpu(), label="SAC mean")
axes[0].legend()
axes[1].plot(h[:, 0], label="Q")
axes[1].plot(h[:, 1], label="policy")
axes[1].legend()
fig.tight_layout()
plt.show()

maximum-entropy objective는 reward뿐 아니라 entropy도 보상합니다. alpha>0이면 약간 낮은 Q action에도 확률을 남기는 것이 전체 soft objective에 유리할 수 있습니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3.2, Eq. (1)`의 **maximum-entropy objective**
- **핵심 식:**

  $$
  J_\pi=\mathbb E_{s,\epsilon}[\alpha\log\pi_\theta(a|s)-Q_\phi(s,a)],\qquad a=\tanh(\mu_\theta(s)+\sigma_\theta(s)\epsilon)
  $$

- **입출력 shape:** `one-hot state [B, S] + action [B, 1] → Q [B, 1]; logp [B]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `SoftActorCriticAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. reward와 expected policy entropy를 함께 최대화한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 MuJoCo continuous control과 twin-style off-policy updates를 평가한다. 축소판은 one-step continuous contextual control로 entropy/Q trade-off를 격리한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.